In [ ]:
"""LBCO/HRPT structure Rietveld fit — a diffraction-lib tutorial port."""

# Structure Refinement: LBCO, HRPT

This minimalistic example is designed to show how Rietveld refinement
can be performed when both the crystal structure and experiment are
defined directly in code. Only the experimentally measured data is
loaded from an external file. It also shows how to switch calculation
engine.

For this example, constant-wavelength neutron powder diffraction data
for La0.5Ba0.5CoO3 from HRPT at PSI is used.

It does not contain any advanced features or options, and includes no
comments or explanations — these can be found in the other tutorials.
Default values are used for all parameters if not specified. Only
essential and self-explanatory code is provided.

The example is intended for users who are already familiar with the
EasyDiffraction library and want to quickly get started with a simple
refinement. It is also useful for those who want to see what a
refinement might look like in code. For a more detailed explanation of
the code, please refer to the other tutorials.

> ⛔ **Execution-disabled**: this tutorial is committed ahead of the API it
> exercises and stays out of notebook execution until **tutorial-runner** lands — see
> `execution-exclusions.yml`, whose list shrinks to empty as unblocking tasks land.

## 🛠️ Import Library

In [ ]:
import edi

## 📦 Define Project

In [ ]:
project = edi.Project(name='lbco_hrpt')

## 🧩 Define Structure

In [ ]:
project.structures.create(name='lbco')

In [ ]:
structure = project.structures['lbco']

In [ ]:
structure.space_group.name_h_m = 'P m -3 m'
structure.space_group.coord_system_code = '1'

In [ ]:
structure.cell.length_a = 3.88

In [ ]:
structure.atom_sites.create(
    id='La',
    type_symbol='La',
    fract_x=0,
    fract_y=0,
    fract_z=0,
    adp_iso=0.5,
    occupancy=0.5,
)
structure.atom_sites.create(
    id='Ba',
    type_symbol='Ba',
    fract_x=0,
    fract_y=0,
    fract_z=0,
    adp_iso=0.5,
    occupancy=0.5,
)
structure.atom_sites.create(
    id='Co',
    type_symbol='Co',
    fract_x=0.5,
    fract_y=0.5,
    fract_z=0.5,
    adp_iso=0.5,
)
structure.atom_sites.create(
    id='O',
    type_symbol='O',
    fract_x=0,
    fract_y=0.5,
    fract_z=0.5,
    adp_iso=0.5,
)

In [ ]:
project.display.structure(struct_name='lbco')

## 🔬 Define Experiment

In [ ]:
data_path = edi.download_data('meas-lbco-hrpt', destination='data')

In [ ]:
project.experiments.add_from_data_path(
    name='hrpt',
    data_path=data_path,
    sample_form='powder',
    beam_mode='constant wavelength',
    radiation_probe='neutron',
)

In [ ]:
experiment = project.experiments['hrpt']

In [ ]:
experiment.instrument.setup_wavelength = 1.494
experiment.instrument.calib_twotheta_offset = 0.6

In [ ]:
experiment.peak.broad_gauss_u = 0.1
experiment.peak.broad_gauss_v = -0.1
experiment.peak.broad_gauss_w = 0.1
experiment.peak.broad_lorentz_y = 0.1

In [ ]:
experiment.excluded_regions.create(id='1', start=0, end=5)
experiment.excluded_regions.create(id='2', start=165, end=180)

In [ ]:
experiment.background.auto_estimate()

In [ ]:
experiment.linked_structures.create(structure_id='lbco', scale=10.0)

## 🚀 Perform Analysis

### Without Constraints

In [ ]:
structure.cell.length_a.free = True

structure.atom_sites['La'].adp_iso.free = True
structure.atom_sites['Ba'].adp_iso.free = True
structure.atom_sites['Co'].adp_iso.free = True
structure.atom_sites['O'].adp_iso.free = True

In [ ]:
experiment.instrument.calib_twotheta_offset.free = True

experiment.peak.broad_gauss_u.free = True
experiment.peak.broad_gauss_v.free = True
experiment.peak.broad_gauss_w.free = True
experiment.peak.broad_lorentz_y.free = True

for point in experiment.background:
    point.intensity.free = True

experiment.linked_structures['lbco'].scale.free = True

In [ ]:
project.analysis.fit()

In [ ]:
project.display.fit.results()

In [ ]:
# project.display.fit.correlations()
# ⛔ WIP PLACEHOLDER — not yet in edi
print('[WIP] project.display.fit.correlations is not implemented in edi yet — skipped')

In [ ]:
project.display.pattern(expt_name='hrpt')

### With Constraints

In [ ]:
# As can be seen from the parameter-correlation plot, the isotropic
# displacement parameters of La and Ba are highly correlated. Because
# La and Ba share the same mixed-occupancy site, their contributions to
# the neutron diffraction pattern are difficult to separate, especially
# since their coherent scattering lengths are not very different.
# Therefore, it is necessary to constrain them to be equal. First we
# define aliases and then use them to create a constraint.
project.analysis.aliases.create(
    id='biso_La',
    param=project.structures['lbco'].atom_sites['La'].adp_iso,
)
project.analysis.aliases.create(
    id='biso_Ba',
    param=project.structures['lbco'].atom_sites['Ba'].adp_iso,
)
project.analysis.constraints.create(expression='biso_Ba = biso_La')

In [ ]:
# project.analysis.minimizer.show_supported()
# project.analysis.minimizer.type = 'lmfit'
# ⛔ TASK-SCOPED PLACEHOLDER — the minimizer and calculator selectors are DEFERRED to a
# later milestone (owner
# ruling 2026-09-22, unfiled at porting time); this placeholder expires when that milestone lands.
print('[PLACEHOLDER] the fitting-backend selector is deferred to a later milestone — skipped')

In [ ]:
project.analysis.fit()

In [ ]:
project.display.fit.results()

In [ ]:
# project.display.fit.correlations()
# ⛔ WIP PLACEHOLDER — not yet in edi
print('[WIP] project.display.fit.correlations is not implemented in edi yet — skipped')

In [ ]:
project.display.pattern(expt_name='hrpt')

### Switch Calculator

In [ ]:
# experiment.calculator.show_supported()
# ⛔ TASK-SCOPED PLACEHOLDER — the calculator and minimizer selectors are DEFERRED to a
# later milestone (owner
# ruling 2026-09-22, unfiled at porting time); this placeholder expires when that milestone lands.
print('[PLACEHOLDER] the pattern-engine selector is deferred to a later milestone — skipped')

In [ ]:
# experiment.calculator.type = 'crysfml'
# ⛔ TASK-SCOPED PLACEHOLDER — the calculator and minimizer selectors are DEFERRED to a
# later milestone (owner
# ruling 2026-09-22, unfiled at porting time); this placeholder expires when that milestone lands.
print('[PLACEHOLDER] the pattern-engine selector is deferred to a later milestone — skipped')

In [ ]:
project.analysis.fit()

In [ ]:
project.display.fit.results()

In [ ]:
# project.display.fit.correlations()
# ⛔ WIP PLACEHOLDER — not yet in edi
print('[WIP] project.display.fit.correlations is not implemented in edi yet — skipped')

In [ ]:
project.display.pattern(expt_name='hrpt')

## 💾 Save Project

In [ ]:
project.save_as(dir_path='projects/refine-lbco-hrpt-from-data')